In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, r2_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_selector,make_column_transformer
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline
from tqdm import tqdm

In [5]:
insurance = pd.read_csv("F:\MACHINE LEARNING\SessionsMaterial\Cases\Medical_Cost_Personal\insurance.csv")
X,y=insurance.drop('charges',axis=1),insurance['charges']
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=26,test_size=.3)
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas')
ct=make_column_transformer((ohe,make_column_selector(dtype_include=object)),('passthrough',make_column_selector(dtype_exclude=object)),verbose_feature_names_out=False).set_output(transform='pandas')


In [8]:

rates=np.linspace(0.01,.9,10)
depth=[None,2,3,4]
n_est=[25,50,100]
scores=[]
for r in tqdm(rates):
    for n in n_est:
        for d in depth:
            gdm=GradientBoostingRegressor(random_state=26,max_depth=d,n_estimators=n,learning_rate=r)
            pipe=Pipeline([('ct',ct),('gdm',gdm)])
            pipe.fit(X_train, y_train)
            pred = pipe.predict(X_test)
            scores.append([r,d,n, r2_score(y_test, pred)])
score_df=pd.DataFrame(scores,columns=["learning_rate",'max_depth','n_estimators','r2_score(' ])
score_df.sort_values(by="r2_score(",ascending=False,ignore_index=True)[:20]

100%|██████████| 10/10 [00:15<00:00,  1.52s/it]


,learning_rate,max_depth,n_estimators,r2_score(
0,0.108889,3.0,50,0.869232
1,0.207778,3.0,25,0.869119
2,0.108889,3.0,100,0.868956
3,0.108889,3.0,25,0.866720
4,0.306667,3.0,25,0.866690
5,0.207778,3.0,50,0.866681
6,0.405556,2.0,50,0.866559
7,0.108889,4.0,50,0.866489
8,0.108889,4.0,25,0.865897
9,0.108889,2.0,100,0.864991


In [21]:
learn_r = np.linspace(0.01, 0.9, 10)
max_depth = [None, 2,3,4]
n_est = [25,50,100]
scores = []

for r in tqdm(learn_r):
    for m in max_depth:
        for n in n_est:
            gbm = CatBoostRegressor(random_state=26, learning_rate=r, max_depth= m, n_estimators=n, cat_features=['sex', 'smoker', 'region'] ,verbose = False)
            gbm.fit(X_train, y_train)
            pred = gbm.predict(X_test)
            scores.append([r,m,n, r2_score(y_test, pred)])
score_df=pd.DataFrame(scores,columns=["learning_rate",'max_depth','n_estimators','accuracy_score(' ])
score_df.sort_values(by="accuracy_score(",ascending=False,ignore_index=True)[:20]

100%|██████████| 10/10 [01:53<00:00, 11.40s/it]


,learning_rate,max_depth,n_estimators,accuracy_score(
0,0.108889,3.0,100,0.873361
1,0.108889,4.0,100,0.871817
2,0.306667,3.0,25,0.871504
3,0.306667,NaN,25,0.871146
4,0.108889,NaN,100,0.871091
5,0.108889,3.0,50,0.871087
6,0.306667,3.0,50,0.870750
7,0.108889,4.0,50,0.870548
8,0.207778,3.0,50,0.870478
9,0.207778,4.0,50,0.870296
